In [ ]:
import json,pickle,zipfile,re,math,os,time,copy,inspect,unicodedata
import numpy as np
import unicodedata
import torch, torch.nn as nn, torch.nn.functional as F
from collections import Counter
from tqdm import tqdm
from tqdm.auto import tqdm
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient
from torch.utils.data import DataLoader
from torch.utils import checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

In [ ]:
## tokenizer
 
WORD_END = 256
 
def pre_tokenize_preserve(text):
    return re.findall(r"\w+|[^\w\s]|\s+", text)
 
def word_to_symbols(word):
    byte_values = word.encode("utf-8")
    symbols = [(b,) for b in byte_values]
    symbols.append((WORD_END,))
    return tuple(symbols)
 
def get_adjacent_pairs(symbols):
    return set(zip(symbols, symbols[1:]))
 
def apply_bpe_merges(symbols, merge_rank):
    symbols = list(symbols)
    while len(symbols) > 1:
        pairs = get_adjacent_pairs(symbols)
        candidates = [p for p in pairs if p in merge_rank]
        if not candidates:
            break
        best_pair = min(candidates, key=lambda p: merge_rank[p])
        sa, sb = best_pair
        merged = sa + sb
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == sa and symbols[i + 1] == sb:
                new_symbols.append(merged)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols
 
def load_tokenizer(pkl_path):
    bpe_data = pickle.load(open(pkl_path, "rb"))
    stoi, merge_rank = bpe_data["stoi"], bpe_data["merge_rank"]
    cache = {}
 
    def encode_unit(unit):
        if unit not in cache:
            cache[unit] = [stoi[s] for s in apply_bpe_merges(word_to_symbols(unit), merge_rank)]
        return cache[unit]
 
    def encode(text):
        ids = []
        for unit in pre_tokenize_preserve(text):
            ids += encode_unit(" " if unit.isspace() else unit)
        return ids
 
    return bpe_data, encode
 
def clean_text(t):
    t = unicodedata.normalize("NFKC", t)
    t = re.sub(r"(\w)-[ \t]*\n[ \t]*(\w)", r"\1\2", t)
    return re.sub(r"\s+", " ", t).strip()

# MODEL V2

In [ ]:
# --------------------------------------------------------------------------
# Rotary Positional Embeddings
# --------------------------------------------------------------------------

class RotaryEmbedding(nn.Module):
    """Precomputes cos/sin tables and applies RoPE to q/k tensors."""

    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0, "RoPE requires an even head_dim"
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.max_seq_len = max_seq_len
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        """Returns cos, sin of shape (seq_len, head_dim) for positions [offset, offset+seq_len)."""
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    """x: (B, n_head, T, head_dim); cos/sin: (T, head_dim)"""
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)


# --------------------------------------------------------------------------
# Attention / MLP / Block
# --------------------------------------------------------------------------

class CausalSelfAttention(nn.Module):
    """Multi-head self-attention with a causal mask and RoPE (instead of
    relying on absolute positional embeddings added at the input). Uses
    torch's fused scaled_dot_product_attention kernel -- a standard
    PyTorch primitive, not a pretrained component."""

    def __init__(self, n_embd, n_head, context_length, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.n_embd = n_embd
        self.head_dim = n_embd // n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.context_length = context_length
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # Apply RoPE to q/k at their true positions before caching/concatenation --
        # position_offset lets a single cached token be rotated to its real index.
        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present = (k, v) if use_cache else None

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if attn_mask is not None:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=dropout_p)
        elif past_kv is not None:
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=dropout_p)
        else:
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=dropout_p, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    """4x-expansion feedforward block with GELU, as in GPT-2."""

    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    """One transformer block, pre-LN style: x = x + Attn(LN(x)); x = x + MLP(LN(x))."""

    def __init__(self, n_embd, n_head, context_length, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, context_length, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        attn_out, present = self.attn(self.ln_1(x), attn_mask, past_kv, use_cache, position_offset)
        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))
        return (x, present) if use_cache else x


# --------------------------------------------------------------------------
# Full model
# --------------------------------------------------------------------------

class DecoderOnlyLM_RoPE(nn.Module):
    """GPT-2-style decoder-only LM using RoPE instead of a learned absolute
    positional embedding table. No wpe -- positions are injected inside
    attention via rotation of q/k, so they naturally extrapolate a little
    beyond context_length and don't need a fixed lookup table sized to it."""

    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head,
                dropout=0.1, pad_id=0, gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.gradient_checkpointing = gradient_checkpointing
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, context_length, dropout) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            nn.init.zeros_(module.bias)
            nn.init.ones_(module.weight)

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        combined_mask = None
        if attention_mask is not None:
            key_mask = attention_mask.bool()[:, None, None, :]
            if past_len == 0:
                causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device))
                eye = torch.eye(T, dtype=torch.bool, device=input_ids.device)
                combined_mask = (causal[None, None] & key_mask) | eye[None, None]
            else:
                combined_mask = key_mask

        # No position_ids/wpe lookup needed -- RoPE injects position inside attention,
        # via position_offset (= past_len) passed to each block below.
        x = self.drop(self.wte(input_ids))

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = past_key_values[i] if past_key_values is not None else None
            if self.gradient_checkpointing and self.training and not use_cache:
                x = checkpoint.checkpoint(block, x, combined_mask, None, False, past_len,
                                          use_reentrant=False)
            else:
                out = block(x, combined_mask, past, use_cache, past_len)
                if use_cache:
                    x, present = out
                    presents.append(present)
                else:
                    x = out

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=attention_mask,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


# # ---- Instantiate for this run ----
# model = DecoderOnlyLM_RoPE(
#     vocab_size=VOCAB_SIZE,          # from your bpe_data["vocab_size"]
#     context_length=1536,
#     n_embd=640,
#     n_layer=10,
#     n_head=10,
#     pad_id=PAD_ID,                  # from your bpe_data["stoi"]["<pad>"]
#     gradient_checkpointing=True,
# )

# n_params = count_parameters(model)
# print(f"Model size: {n_params/1e6:.2f}M parameters")

# MODEL V5

In [ ]:
# --------------------------------------------------------------------------
# RMSNorm
# --------------------------------------------------------------------------

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm_x = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm_x * self.weight


# --------------------------------------------------------------------------
# RoPE (back from the ALiBi swap)
# --------------------------------------------------------------------------

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)


# --------------------------------------------------------------------------
# Grouped Query Attention (unchanged)
# --------------------------------------------------------------------------

class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = n_embd // n_head

        self.q_proj = nn.Linear(n_embd, n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)

        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present = (k, v) if use_cache else None

        k_exp = k.repeat_interleave(self.n_rep, dim=1)
        v_exp = v.repeat_interleave(self.n_rep, dim=1)

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if attn_mask is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)
        elif past_kv is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p)
        else:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


# --------------------------------------------------------------------------
# Mixture-of-Experts feedforward (Switch-Transformer style, top-1 routing).
# Instead of one dense FFN processing every token, several smaller expert
# FFNs exist, and a router sends each token to exactly ONE expert. Total
# model capacity grows with num_experts, but compute per token stays close
# to a single expert's cost -- the "efficient" lever here is more capacity
# for about the same FLOPs per token, not less compute overall.
# --------------------------------------------------------------------------

class Expert(nn.Module):
    def __init__(self, n_embd, expert_hidden_mult=4, dropout=0.1):
        super().__init__()
        hidden = expert_hidden_mult * n_embd
        self.c_fc = nn.Linear(n_embd, hidden)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(hidden, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class MoEFeedForward(nn.Module):
    def __init__(self, n_embd, num_experts=4, expert_hidden_mult=4, dropout=0.1):
        super().__init__()
        self.num_experts = num_experts
        self.router = nn.Linear(n_embd, num_experts, bias=False)
        self.experts = nn.ModuleList([
            Expert(n_embd, expert_hidden_mult, dropout) for _ in range(num_experts)
        ])

    def forward(self, x):
        """x: (B, T, C). Returns (output, aux_loss). aux_loss is the
        Switch-Transformer load-balancing term: it penalizes the router for
        sending a disproportionate share of tokens to any one expert, which
        otherwise collapses to using just 1-2 experts and wasting the rest
        of the model's capacity.

        Implementation note: this runs EVERY expert on EVERY token (dense),
        then zeros out all but the top-1 selection via a one-hot weight,
        rather than using masked indexing (x_flat[mask]). Masked indexing
        produces data-dependent tensor shapes, which breaks gradient
        checkpointing's recomputation check (the recomputed forward pass
        can route tokens slightly differently due to upstream dropout,
        producing mismatched shapes vs. the original forward). This dense
        version costs more FLOPs per token than true sparse routing, but
        every intermediate tensor has a FIXED shape, so it's safe to
        checkpoint."""
        B, T, C = x.shape
        x_flat = x.view(-1, C)                                  # (N, C), N = B*T

        router_logits = self.router(x_flat)                     # (N, E)
        router_probs = F.softmax(router_logits, dim=-1)
        top1_prob, top1_idx = router_probs.max(dim=-1)           # (N,), (N,)
        one_hot = F.one_hot(top1_idx, num_classes=self.num_experts).to(x.dtype)   # (N, E)

        expert_outputs = torch.stack([expert(x_flat) for expert in self.experts], dim=1)  # (N, E, C)
        gate = (one_hot * top1_prob.unsqueeze(-1)).unsqueeze(-1)                          # (N, E, 1)
        output = (expert_outputs * gate).sum(dim=1)                                        # (N, C)

        # Load-balancing aux loss: E * sum_e( fraction_of_tokens_to_e * mean_router_prob_for_e )
        # Minimized (-> 1.0, its floor) when routing is perfectly uniform.
        with torch.no_grad():
            tokens_per_expert = one_hot.float().mean(dim=0)      # (E,)
        mean_router_prob = router_probs.mean(dim=0)              # (E,)
        aux_loss = self.num_experts * (tokens_per_expert * mean_router_prob).sum()

        return output.view(B, T, C), aux_loss


# --------------------------------------------------------------------------
# Sandwich-Norm block: normalizes both BEFORE and AFTER each sublayer,
# instead of only before (plain pre-norm). The extra post-sublayer norm
# bounds how much any single sublayer's output can grow before it's added
# back into the residual stream -- helps control activation growth in
# deeper stacks, at the cost of two extra norm layers per block.
# --------------------------------------------------------------------------

class Block(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, num_experts, dropout=0.1):
        super().__init__()
        self.ln_1_pre = RMSNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, context_length, dropout)
        self.ln_1_post = RMSNorm(n_embd)

        self.ln_2_pre = RMSNorm(n_embd)
        self.moe = MoEFeedForward(n_embd, num_experts=num_experts, dropout=dropout)
        self.ln_2_post = RMSNorm(n_embd)

    def forward_train(self, x, attn_mask):
        """Training path: no cache, returns (x, aux_loss) -- both tensors,
        so this is directly compatible with gradient checkpointing."""
        attn_out, _ = self.attn(self.ln_1_pre(x), attn_mask)
        x = x + self.ln_1_post(attn_out)

        moe_out, aux_loss = self.moe(self.ln_2_pre(x))
        x = x + self.ln_2_post(moe_out)
        return x, aux_loss

    def forward_infer(self, x, past_kv, use_cache, position_offset):
        """Generation path: no attn_mask needed (handled via past_kv/is_causal
        inside attention), no aux_loss (it's a training-only regularizer)."""
        attn_out, present = self.attn(self.ln_1_pre(x), None, past_kv, use_cache, position_offset)
        x = x + self.ln_1_post(attn_out)

        moe_out, _ = self.moe(self.ln_2_pre(x))
        x = x + self.ln_2_post(moe_out)
        return x, present


# --------------------------------------------------------------------------
# Full model
# --------------------------------------------------------------------------

class DecoderOnlyLM_MoE_Sandwich(nn.Module):
    """RoPE + GQA backbone, Sandwich-Norm (RMSNorm before AND after each
    sublayer), and a top-1 Mixture-of-Experts feedforward in place of a
    single dense FFN. aux_loss_weight controls how strongly the
    load-balancing term is weighted into the total loss."""

    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head, n_kv_head,
                num_experts=4, aux_loss_weight=0.01, dropout=0.1, pad_id=0,
                gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.aux_loss_weight = aux_loss_weight
        self.gradient_checkpointing = gradient_checkpointing

        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, n_kv_head, context_length, num_experts, dropout)
            for _ in range(n_layer)
        ])
        self.ln_f = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        x = self.drop(self.wte(input_ids))

        if use_cache:
            # Generation path -- no attn_mask building, no aux_loss tracking.
            presents = []
            for i, block in enumerate(self.blocks):
                past = past_key_values[i] if past_key_values is not None else None
                x, present = block.forward_infer(x, past, use_cache, past_len)
                presents.append(present)
        else:
            # Training path -- build the causal(+padding) mask once, track aux losses.
            combined_mask = None
            if attention_mask is not None:
                key_mask = attention_mask.bool()[:, None, None, :]
                causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device))
                eye = torch.eye(T, dtype=torch.bool, device=input_ids.device)
                combined_mask = (causal[None, None] & key_mask) | eye[None, None]

            aux_losses = []
            for block in self.blocks:
                if self.gradient_checkpointing and self.training:
                    x, aux = checkpoint.checkpoint(block.forward_train, x, combined_mask,
                                                   use_reentrant=False)
                else:
                    x, aux = block.forward_train(x, combined_mask)
                aux_losses.append(aux)
            presents = None

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            ce_loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
            aux_loss = torch.stack(aux_losses).mean() if not use_cache else torch.tensor(0.0, device=x.device)
            loss = ce_loss + self.aux_loss_weight * aux_loss

        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=None,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


def count_active_parameters(model, num_experts):
    """Approximate parameters actually USED per token (top-1 routing means
    only 1 of num_experts experts fires), vs. total stored parameters."""
    total = count_parameters(model)
    expert_params = sum(p.numel() for p in model.blocks[0].moe.experts[0].parameters())
    num_layers = len(model.blocks)
    unused_experts_params = expert_params * (num_experts - 1) * num_layers
    return total - unused_experts_params


# ---- Instantiate for this run ----
# model = DecoderOnlyLM_MoE_Sandwich(
#     vocab_size=VOCAB_SIZE,          # from your bpe_data["vocab_size"]
#     context_length=1536,
#     n_embd=640,
#     n_layer=10,
#     n_head=10,
#     n_kv_head=2,
#     num_experts=4,
#     aux_loss_weight=0.01,
#     pad_id=PAD_ID,                  # from your bpe_data["stoi"]["<pad>"]
#     gradient_checkpointing=True,
# )

# n_total = count_parameters(model)
# n_active = count_active_parameters(model, num_experts=4)
# print(f"Total parameters: {n_total/1e6:.2f}M | Active per token (top-1 routing): {n_active/1e6:.2f}M")

# MODEL V6

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm_x = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm_x * self.weight


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)


def is_global_layer(layer_idx, n_layer, global_every_n):
    """Uniform spacing anchored at the LAST layer, so the final layer is
    always global (full context right before the LM head), with every
    global_every_n-th layer before it also global. E.g. n_layer=10,
    global_every_n=4 -> global layers at indices {1, 5, 9} (0-indexed)."""
    return (n_layer - 1 - layer_idx) % global_every_n == 0


# --------------------------------------------------------------------------
# Grouped Query Attention with an optional sliding window. window_size=None
# means this layer is "global" (ordinary full causal attention, exactly as
# in the RoPE+GQA+RMSNorm baseline). A finite window_size means this layer
# only ever attends to the most recent window_size keys -- including at
# generation time, where the KV cache for this layer is trimmed every step
# rather than growing without bound.
# --------------------------------------------------------------------------

class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, window_size=None, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = n_embd // n_head
        self.window_size = window_size   # None = global/full causal layer

        self.q_proj = nn.Linear(n_embd, n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)

        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def _build_mask(self, T, device, key_padding_mask=None):
        i = torch.arange(T, device=device)
        if self.window_size is None:
            causal = i[:, None] >= i[None, :]                              # plain causal
        else:
            diff = i[:, None] - i[None, :]
            causal = (diff >= 0) & (diff < self.window_size)               # causal + bounded window
        mask = causal[None, None, :, :]
        if key_padding_mask is not None:
            key_mask = key_padding_mask.bool()[:, None, None, :]
            eye = torch.eye(T, dtype=torch.bool, device=device)
            mask = (mask & key_mask) | eye[None, None, :, :]
        return mask

    def forward(self, x, key_padding_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
            # Sliding-window layers never need more than window_size keys --
            # trimming here is what actually bounds this layer's cache size
            # at generation time, unlike the global layers' cache which grows
            # with every step exactly like the baseline model.
            if self.window_size is not None and k.size(2) > self.window_size:
                k = k[:, :, -self.window_size:, :]
                v = v[:, :, -self.window_size:, :]
                # key_padding_mask (passed in covering the FULL past_len+T
                # span) must be trimmed to match this layer's shorter cache,
                # or it mismatches k/v's sequence length at the mask step below.
                if key_padding_mask is not None:
                    key_padding_mask = key_padding_mask[:, -self.window_size:]
        present = (k, v) if use_cache else None

        k_exp = k.repeat_interleave(self.n_rep, dim=1)
        v_exp = v.repeat_interleave(self.n_rep, dim=1)

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if past_kv is None and T > 1:
            # Full sequence, first call: build the real causal(+window) mask.
            attn_mask = self._build_mask(T, x.device, key_padding_mask)
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)
        else:
            # Decode step: single new query, already-trimmed cache -- every
            # remaining cached key is, by construction, within the window
            # (or there is no window for a global layer), so no extra mask
            # is needed beyond optional padding exclusion.
            attn_mask = None
            if key_padding_mask is not None:
                attn_mask = key_padding_mask.bool()[:, None, None, :]
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    """Sandwich-Norm: RMSNorm both BEFORE and AFTER each sublayer, instead
    of only before (plain pre-norm). The post-sublayer norm bounds how much
    the attention/MLP output can grow before it's added back into the
    residual stream -- four RMSNorms per block instead of two."""

    def __init__(self, n_embd, n_head, n_kv_head, context_length, window_size, dropout=0.1):
        super().__init__()
        self.ln_1_pre = RMSNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, context_length, window_size, dropout)
        self.ln_1_post = RMSNorm(n_embd)
        self.ln_2_pre = RMSNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)
        self.ln_2_post = RMSNorm(n_embd)

    def forward(self, x, key_padding_mask=None, past_kv=None, use_cache=False, position_offset=0):
        attn_out, present = self.attn(self.ln_1_pre(x), key_padding_mask, past_kv, use_cache, position_offset)
        x = x + self.ln_1_post(attn_out)
        x = x + self.ln_2_post(self.mlp(self.ln_2_pre(x)))
        return (x, present) if use_cache else x


# --------------------------------------------------------------------------
# Full model
# --------------------------------------------------------------------------

class DecoderOnlyLM_HybridMask(nn.Module):
    """RoPE + GQA backbone with two changes versus the proven #4 model:
    (1) Sandwich-Norm -- RMSNorm both before and after each sublayer,
    instead of only before; (2) layers alternate between full-causal
    ("global") and sliding-window ("local") attention at a fixed, uniform
    spacing through depth -- e.g. every 4th layer is global, the rest are
    local. Combining both in one model means a BERTScore difference versus
    #4 can't be cleanly attributed to just one change -- if you want a
    single-variable comparison, train a masking-only version (plain
    pre-norm) and a norm-only version (full-causal + sandwich-norm)
    alongside this one."""

    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head, n_kv_head,
                window_size=192, global_every_n=4, dropout=0.1, pad_id=0,
                gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.gradient_checkpointing = gradient_checkpointing
        self.window_size = window_size
        self.global_every_n = global_every_n

        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)

        layer_windows = [
            None if is_global_layer(i, n_layer, global_every_n) else window_size
            for i in range(n_layer)
        ]
        print(f"Layer attention pattern (None = global/full-causal): {layer_windows}")

        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, n_kv_head, context_length, layer_windows[i], dropout)
            for i in range(n_layer)
        ])
        self.ln_f = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        if attention_mask is None and past_key_values is None:
            attention_mask = input_ids.ne(self.pad_id)

        x = self.drop(self.wte(input_ids))

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = past_key_values[i] if past_key_values is not None else None
            if self.gradient_checkpointing and self.training and not use_cache:
                x = checkpoint.checkpoint(block, x, attention_mask, None, False, past_len,
                                          use_reentrant=False)
            else:
                out = block(x, attention_mask, past, use_cache, past_len)
                if use_cache:
                    x, present = out
                    presents.append(present)
                else:
                    x = out

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]
        running_mask = attention_mask

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)
            running_mask = torch.cat([running_mask, torch.ones_like(next_token)], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=running_mask,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


# ---- Instantiate for this run ----
# n_layer=10, global_every_n=4 -> global (full-causal) layers at indices
# {1, 5, 9}; the rest use a 192-token sliding window. The cache for those
# 7 local layers never exceeds 192 entries at generation time, regardless
# of how many tokens have been generated.
# model = DecoderOnlyLM_HybridMask(
#     vocab_size=VOCAB_SIZE,          # from your bpe_data["vocab_size"]
#     context_length=1536,
#     n_embd=640,
#     n_layer=10,
#     n_head=10,
#     n_kv_head=2,
#     window_size=192,
#     global_every_n=4,
#     pad_id=PAD_ID,                  # from your bpe_data["stoi"]["<pad>"]
#     gradient_checkpointing=True,
# )

# n_params = count_parameters(model)
# print(f"Model size: {n_params/1e6:.2f}M parameters")

# MODEL V7

In [ ]:
# --------------------------------------------------------------------------
# RMSNorm
# --------------------------------------------------------------------------

class RMSNorm(nn.Module):
    """Root-mean-square norm: rescales by RMS of activations, no mean
    subtraction and no bias, unlike LayerNorm. Cheaper, and what most
    recent open models (LLaMA-style) use instead of LayerNorm."""

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm_x = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm_x * self.weight


# --------------------------------------------------------------------------
# Rotary Positional Embeddings (unchanged from the RoPE variant)
# --------------------------------------------------------------------------

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0, "RoPE requires an even head_dim"
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.max_seq_len = max_seq_len
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    """x: (B, n_heads, T, head_dim); cos/sin: (T, head_dim). Works for both
    query heads and kv heads since head_dim is shared across them."""
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)


# --------------------------------------------------------------------------
# Grouped Query Attention
# --------------------------------------------------------------------------

class GroupedQueryAttention(nn.Module):
    """Like standard multi-head self-attention, but n_kv_head < n_head:
    groups of query heads share the same key/value heads. Cuts the KV
    cache size (and the K/V projection cost) by a factor of n_head/n_kv_head,
    with little quality loss in practice -- the point of GQA."""

    def __init__(self, n_embd, n_head, n_kv_head, context_length, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0, "n_head must be divisible by n_kv_head"
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = n_embd // n_head

        self.q_proj = nn.Linear(n_embd, n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)

        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.context_length = context_length
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            # cache stores the SMALL n_kv_head-sized k/v -- this is where GQA
            # actually saves memory during generation, not just compute.
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present = (k, v) if use_cache else None

        # Expand kv heads to match query heads right before attention.
        k_exp = k.repeat_interleave(self.n_rep, dim=1)
        v_exp = v.repeat_interleave(self.n_rep, dim=1)

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if attn_mask is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)
        elif past_kv is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p)
        else:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    """4x-expansion feedforward block with GELU, as in GPT-2."""

    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    """Pre-norm block using RMSNorm: x = x + Attn(RMSNorm(x)); x = x + MLP(RMSNorm(x))."""

    def __init__(self, n_embd, n_head, n_kv_head, context_length, dropout=0.1):
        super().__init__()
        self.ln_1 = RMSNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, context_length, dropout)
        self.ln_2 = RMSNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        attn_out, present = self.attn(self.ln_1(x), attn_mask, past_kv, use_cache, position_offset)
        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))
        return (x, present) if use_cache else x


# --------------------------------------------------------------------------
# Full model
# --------------------------------------------------------------------------

class DecoderOnlyLM_RMSNorm_GQA(nn.Module):
    """Decoder-only LM: RMSNorm (pre-norm) + Grouped Query Attention + RoPE.
    No wpe -- positions come from RoPE inside attention, as in the RoPE
    variant. n_kv_head < n_head is what makes this GQA; n_kv_head == n_head
    would reduce to standard multi-head attention."""

    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head, n_kv_head,
                dropout=0.1, pad_id=0, gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.gradient_checkpointing = gradient_checkpointing
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, n_kv_head, context_length, dropout) for _ in range(n_layer)
        ])
        self.ln_f = RMSNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        # RMSNorm's weight is already initialized to ones in its own __init__.

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        combined_mask = None
        if attention_mask is not None:
            key_mask = attention_mask.bool()[:, None, None, :]
            if past_len == 0:
                causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device))
                eye = torch.eye(T, dtype=torch.bool, device=input_ids.device)
                combined_mask = (causal[None, None] & key_mask) | eye[None, None]
            else:
                combined_mask = key_mask

        x = self.drop(self.wte(input_ids))

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = past_key_values[i] if past_key_values is not None else None
            if self.gradient_checkpointing and self.training and not use_cache:
                x = checkpoint.checkpoint(block, x, combined_mask, None, False, past_len,
                                          use_reentrant=False)
            else:
                out = block(x, combined_mask, past, use_cache, past_len)
                if use_cache:
                    x, present = out
                    presents.append(present)
                else:
                    x = out

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=attention_mask,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


# ---- Instantiate for this run ----
# n_kv_head < n_head is what makes this GQA. n_head=12 with n_kv_head=3 means
# each kv head is shared by 5 query heads -- a common ~4-8x ratio in practice.
# model = DecoderOnlyLM_RMSNorm_GQA(
#     vocab_size=VOCAB_SIZE,          # from your bpe_data["vocab_size"]
#     context_length=1536,
#     n_embd=768, # prev value 640 for v4
#     n_layer=10,
#     n_head=12,  # previous one 10 for v4
#     n_kv_head=3,# prev value 2
#     pad_id=PAD_ID,                  # from your bpe_data["stoi"]["<pad>"]
#     gradient_checkpointing=True,
# )

# n_params = count_parameters(model)
# print(f"Model size: {n_params/1e6:.2f}M parameters")

In [ ]:
def build_model():
    # for  v7
    return DecoderOnlyLM_RMSNorm_GQA(
    vocab_size=VOCAB_SIZE,          # from your bpe_data["vocab_size"]
    context_length=1536,
    n_embd=768, # prev value 640 for v4
    n_layer=10,
    n_head=12,  # previous one 10 for v4
    n_kv_head=3,# prev value 2
    pad_id=PAD_ID,                  # from your bpe_data["stoi"]["<pad>"]
    gradient_checkpointing=True,
) 
# for v2
#       return DecoderOnlyLM_RoPE(
#     vocab_size=VOCAB_SIZE,          # from your bpe_data["vocab_size"]
#     context_length=1536,
#     n_embd=640,
#     n_layer=10,
#     n_head=10,
#     pad_id=PAD_ID,                  # from your bpe_data["stoi"]["<pad>"]
#     gradient_checkpointing=True,
# )
# for v5
#       return DecoderOnlyLM_MoE_Sandwich(
#     vocab_size=VOCAB_SIZE,          # from your bpe_data["vocab_size"]
#     context_length=1536,
#     n_embd=640,
#     n_layer=10,
#     n_head=10,
#     n_kv_head=2,
#     num_experts=2,
#     aux_loss_weight=0.01,
#     pad_id=PAD_ID,                  # from your bpe_data["stoi"]["<pad>"]
#     gradient_checkpointing=True,
# )
# for v6
#     return DecoderOnlyLM_HybridMask(
#     vocab_size=VOCAB_SIZE,          # from your bpe_data["vocab_size"]
#     context_length=1536,
#     n_embd=640,
#     n_layer=10,
#     n_head=10,
#     n_kv_head=2,
#     window_size=192,
#     global_every_n=4,
#     pad_id=PAD_ID,                  # from your bpe_data["stoi"]["<pad>"]
#     gradient_checkpointing=True,
# )


HEAD_PATH   = "lm_head"      # dotted path of the LM head in your model ("decoder.lm_head" inside an enc-dec class)
LAYERS_PATH = "blocks"           # block stack, e.g. "layers" / "blocks" / "decoder.layers" (None = auto if unambiguous)
FINAL_NORM  = ("ln_f",)      # name of your final norm module (kept trainable in partial FT)
MODEL_PT    = "/kaggle/working/decoder_baseline_v7_final_model.pt"

In [ ]:
def build_decoder(bpe_data):
    stoi, V = bpe_data["stoi"], bpe_data["vocab_size"]
    id2bytes = [b""] * V
    for sym, i in stoi.items():
        if isinstance(sym, tuple) and 0 <= i < V:
            id2bytes[i] = bytes(b for b in sym if b != WORD_END)

    def decode(ids):
        return b"".join(id2bytes[i] for i in ids if 0 <= i < V).decode("utf-8", errors="replace")
    return decode

DOC_KEYS = ("document", "text", "judgment", "doc")
SUM_KEYS = ("summary", "reference_summary", "reference", "target")

def normalize_pairs(raw):
    if isinstance(raw, dict):
        if isinstance(raw.get("data"), list):
            raw = raw["data"]
        else:
            raw = [{"id": k, **(v if isinstance(v, dict) else {"document": v})} for k, v in raw.items()]
    out = []
    for d in raw:
        doc = next((d[k] for k in DOC_KEYS if k in d), None)
        if doc is None:
            raise ValueError(f"case {d.get('id')!r} has no document field (tried {DOC_KEYS}); keys: {list(d)}")
        if isinstance(doc, list):
            doc = " ".join(str(x) for x in doc)
        summ = next((d[k] for k in SUM_KEYS if k in d), None)
        if isinstance(summ, list):
            summ = " ".join(str(x) for x in summ)
        out.append({"id": str(d["id"]), "document": doc, "summary": summ})
    return out

def summ_paths(root, include_test=False):
    p = {"train": f"{root}/Train/train.json"}
    if include_test:
        p["test"] = f"{root}/Test/test.json"
    return p

def download_summ_files(repo="Exploration-Lab/CS-781-Capstone", p6="Phase_6_Summ", token=None, include_test=False):
   return {k: hf_hub_download(repo_id=repo, filename=f, repo_type="dataset", token=token) for k, f in summ_paths(p6, include_test).items()}

def tokenize_pairs(pairs, encode_fn, clean_fn=None, max_doc_tokens=None, desc="tokenizing"):
    out = []
    for p in tqdm(pairs, desc=desc, leave=False):
        doc = clean_fn(p["document"]) if clean_fn else p["document"]
        ids = np.asarray(encode_fn(doc), dtype=np.int32)
        if max_doc_tokens and len(ids) > max_doc_tokens:
            ids = ids[:max_doc_tokens]
        item = {"id": p["id"], "ids": ids, "n_chars": len(p["document"]), "sum_ids": None, "ref": None}
        if p["summary"] is not None:
            ref = clean_fn(p["summary"]) if clean_fn else p["summary"]
            item["ref"], item["sum_ids"] = ref, np.asarray(encode_fn(ref), dtype=np.int32)
        out.append(item)
    return out

def compute_idf(docs, vocab_size):
    df = np.zeros(vocab_size, dtype=np.float64)
    for d in docs:
        df[np.unique(d["ids"])] += 1
    return (np.log((len(docs) + 1) / (df + 1)) + 1.0).astype(np.float32)

def split_pairs(docs, val_size=100, seed=0):
    perm = np.random.default_rng(seed).permutation(len(docs))
    k = min(val_size, max(1, len(docs) // 10)) if val_size else 0
    val_idx = set(perm[:k].tolist())
    return [d for i, d in enumerate(docs) if i not in val_idx], [docs[i] for i in sorted(val_idx)]

def setup_data(paths, tokenizer_pkl, sep_token="<s>", eos_token="</s>", clean=True, clean_fn=None,val_size=100, seed=0, max_doc_tokens=None, cache=None, include_test=False):
    if isinstance(paths, str):
        paths = summ_paths(paths, include_test)
    bpe_data, encode = load_tokenizer(tokenizer_pkl)
    stoi = bpe_data["stoi"]
    pad_id = stoi["<pad>"]
    print("special tokens:", [t for t in stoi if isinstance(t, str)])
    sep_id, eos_id = stoi.get(sep_token), stoi.get(eos_token)
    if sep_id is None or eos_id is None:
        raise ValueError(f"sep_token {sep_token!r} / eos_token {eos_token!r} must be in the vocabulary (see the list above)")
    clean_fn = (clean_fn or clean_text) if clean else None
    key = {"max_doc_tokens": max_doc_tokens, "clean": bool(clean_fn), "include_test": include_test and "test" in paths}
    cached = None
    if cache and os.path.exists(cache):
        cached = pickle.load(open(cache, "rb"))
        if cached.get("key") != key:
            print("cache settings differ -> re-tokenizing")
            cached = None
        else:
            print(f"loaded tokenized documents from {cache}")
    if cached:
        all_docs, test_docs = cached["train"], cached.get("test")
    else:
        all_docs = tokenize_pairs(normalize_pairs(json.load(open(paths["train"], encoding="utf-8"))), encode, clean_fn,
                                  max_doc_tokens, desc="tokenizing train")
        test_docs = None
        if include_test and "test" in paths:
            test_docs = tokenize_pairs(normalize_pairs(json.load(open(paths["test"], encoding="utf-8"))), encode, clean_fn,
                                       max_doc_tokens, desc="tokenizing test")
        if cache:
            pickle.dump({"key": key, "train": all_docs, "test": test_docs}, open(cache, "wb"))
            print(f"saved tokenized documents to {cache}")

    train, val = split_pairs(all_docs, val_size, seed)
    boundary = set()
    for ch in ".?!;":
        e = encode(ch)
        if len(e) == 1:
            boundary.add(e[0])
    D = {"pad_id": pad_id, "sep_id": sep_id, "eos_id": eos_id, "vocab_size": bpe_data["vocab_size"], "encode": encode,"decode": build_decoder(bpe_data), "clean": clean_fn, "max_doc_tokens": max_doc_tokens, "bpe_data": bpe_data,"train": train, "val": val, "idf": compute_idf(train, bpe_data["vocab_size"]),
         "boundary_ids": np.array(sorted(boundary), dtype=np.int64)}
    if test_docs is not None:
        D["test"] = test_docs
    dl = np.array([len(d["ids"]) for d in all_docs]); sl = np.array([len(d["sum_ids"]) for d in all_docs])
    pct = lambda a, q: int(np.percentile(a, q))
    print(f"{len(train)} train / {len(val)} validation documents")
    print(f"document tokens: median {int(np.median(dl))}, p90 {pct(dl, 90)}, max {dl.max()} | "
          f"summary tokens: median {int(np.median(sl))}, p90 {pct(sl, 90)}, p99 {pct(sl, 99)}, max {sl.max()} | "
          f"compression (median doc / summary) {np.median(dl / np.maximum(sl, 1)):.1f}x")
    print(f"PAD_ID={pad_id}  VOCAB_SIZE={D['vocab_size']}  <sep>={sep_token!r} id {sep_id}  <eos>={eos_token!r} id {eos_id}  "
          f"cleaning={'on' if clean_fn else 'off'}  sentence-boundary token ids {D['boundary_ids'].tolist()}")
    return D

In [ ]:
DOC_HEAD = dict(mode="head", max_len=1536, max_sum_tokens=256)
DOC_TAIL = dict(mode="tail", max_len=1536, max_sum_tokens=256)
DOC_HEADTAIL = dict(mode="headtail", max_len=1536, max_sum_tokens=256, head_frac=0.5)
DOC_SPREAD = dict(mode="spread", max_len=1536, max_sum_tokens=256, windows=4)
DOC_SALIENT = dict(mode="salient", max_len=1536, max_sum_tokens=256, lead_sents=3)


def doc_budget(DOC):
    return DOC["max_len"] - DOC["max_sum_tokens"] - 1


def select_head(ids, n):
    return ids[:n]


def select_tail(ids, n):
    return ids[-n:]


def select_headtail(ids, n, head_frac=0.5):
    if len(ids) <= n:
        return ids
    h = int(n * head_frac)
    return np.concatenate([ids[:h], ids[len(ids) - (n - h):]])


def select_spread(ids, n, windows=4):
    if len(ids) <= n:
        return ids
    w = n // windows
    starts = np.linspace(0, len(ids) - w, windows).round().astype(int)
    return np.concatenate([ids[s:s + w] for s in starts])


def select_salient(ids, n, aux, lead_sents=3):
    if len(ids) <= n:
        return ids
    idf = aux["idf"]
    ends = np.flatnonzero(np.isin(ids, aux["boundary_ids"])) + 1
    cuts = np.unique(np.concatenate([[0], ends, [len(ids)]]))
    sents = [(int(a), int(b)) for a, b in zip(cuts[:-1], cuts[1:]) if b > a]
    doc_vec=doc_tf*idf
    doc_norm=np.linalg.norm(doc_vec) + 1e-8

    scores = []

    for a, b in sents:
        sent_tf = np.bincount(ids[a:b], minlength=len(idf)).astype(np.float32)
        sent_vec = sent_tf * idf
        score=np.dot(sent_vec,doc_vec)/(np.linalg.norm(sent_vec)*doc_norm+1e-8)
    scores.append(score)
    
    chosen, used = set(), 0
    for i in list(range(min(lead_sents, len(sents)))) + [int(j) for j in np.argsort(-scores)]:
        a, b = sents[i]
        if i not in chosen and used + (b - a) <= n:
            chosen.add(i)
            used += b - a
    if not chosen:
        return ids[:n]
    return np.concatenate([ids[sents[i][0]:sents[i][1]] for i in sorted(chosen)])


def select_doc_tokens(ids, DOC, aux=None):
    n, mode = doc_budget(DOC), DOC["mode"]
    if mode == "head":
        return select_head(ids, n)
    if mode == "tail":
        return select_tail(ids, n)
    if mode == "headtail":
        return select_headtail(ids, n, DOC.get("head_frac", 0.5))
    if mode == "spread":
        return select_spread(ids, n, DOC.get("windows", 4))
    if mode == "salient":
        return select_salient(ids, n, aux, DOC.get("lead_sents", 3))
    raise ValueError(f"unknown mode {mode!r}")


def build_examples(docs, DOC, D, with_target=True):
    out = []
    for d in docs:
        prompt = np.append(select_doc_tokens(d["ids"], DOC, D), D["sep_id"]).astype(np.int64)
        ex = {"id": d["id"], "prompt": prompt, "ref": d["ref"], "target": None}
        if with_target:
            ex["target"] = np.append(d["sum_ids"][:DOC["max_sum_tokens"] - 1], D["eos_id"]).astype(np.int64)
        out.append(ex)
    return out


def to_train_items(examples):
    return [{"ids": np.concatenate([e["prompt"], e["target"]]), "n_prompt": len(e["prompt"])} for e in examples]

def collate_summ(batch, pad_id):
    L = max(len(b["ids"]) for b in batch)
    ids = torch.full((len(batch), L), pad_id, dtype=torch.long)
    labels = torch.full((len(batch), L), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        n, k = len(b["ids"]), b["n_prompt"]
        ids[i, :n] = torch.from_numpy(b["ids"])
        labels[i, k:n] = torch.from_numpy(b["ids"][k:])
    return {"input_ids": ids, "labels": labels}

def load_phase1(build_fn, path):
    model = build_fn()
    ckpt = torch.load(path, map_location="cpu")
    sd = ckpt["model_state"] if isinstance(ckpt, dict) and "model_state" in ckpt else ckpt
    model.load_state_dict({k.replace("module.", "", 1): v for k, v in sd.items()}, strict=True)
    return model


def trainable_state(model):
    return {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}


def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    print(f"trainable {tr/1e6:.3f}M / {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")
    return tr, tot


def get_logits(out):
    return out["logits"] if isinstance(out, dict) else out


def lm_loss(model, ids, labels, amp, reduction="mean"):
    with torch.autocast(device_type=ids.device.type, dtype=torch.float16, enabled=amp):
        logits = get_logits(model(ids))
    tgt = labels[:, 1:]
    sel = tgt != -100
    return F.cross_entropy(logits[:, :-1][sel].float(), tgt[sel], reduction=reduction), int(sel.sum())


@torch.no_grad()
def evaluate_loss(model, items, pad_id, device, batch_size=2, use_amp=True):
    model.eval()
    amp = use_amp and device.type == "cuda"
    tot, n = 0.0, 0
    for s in range(0, len(items), batch_size):
        b = collate_summ(items[s:s + batch_size], pad_id)
        loss, k = lm_loss(model, b["input_ids"].to(device), b["labels"].to(device), amp, "sum")
        tot += loss.item(); n += k
    return tot / max(1, n)



def train_summ(model, items, pad_id, cfg, device, val_fn=None, save_path=None, extra_loss=None, on_step=None,resume=False, extra_state=None):
    model.to(device)
    rng = np.random.default_rng(cfg.get("seed", 0))
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    loader = DataLoader(items, batch_size=cfg["batch_size"], shuffle=True, collate_fn=lambda b: collate_summ(b, pad_id))
    total_steps = (math.ceil(len(loader) / cfg["grad_accum"])* cfg["epochs"])
    warmup = int(cfg["warmup_ratio"] * total_steps)

    def lr_lambda(step):
        if step < warmup:
            return step / max(1, warmup)
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    use_amp = cfg["fp16"] and device.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)
    best, history, step, start_epoch = -1e9, [], 0, 0
    last_path = save_path.replace("_best.pt", "_last.pt") if save_path else None
    if resume:
        if last_path and os.path.exists(last_path):
            st = torch.load(last_path, map_location=device)
            model.load_state_dict(st["model"], strict=False)
            opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])
            start_epoch, best, history, step = st["epoch"], st["best"], st["history"], st["step"]
            if st.get("extra") is not None and extra_state is not None:
                extra_state[1](st["extra"])
            print(f"resumed from {last_path}: {start_epoch} epoch(s) already done, best validation score {best:.4f}")
        else:
            print(f"resume=True but {last_path} does not exist -> starting from scratch")
    for epoch in range(start_epoch, cfg["epochs"]):
        model.train()
        run, nb, t0 = 0.0, 0, time.time()
        opt.zero_grad()
        pbar = tqdm(loader, desc=f"epoch {epoch + 1}/{cfg['epochs']}", leave=False)
        for i, b in enumerate(pbar):
            loss, _ = lm_loss(model, b["input_ids"].to(device), b["labels"].to(device), use_amp)
            if extra_loss is not None:
                loss = loss + extra_loss(model)
            scaler.scale(loss / cfg["grad_accum"]).backward()
            run += loss.item(); nb += 1
            pbar.set_postfix(loss=f"{run / nb:.4f}")
            if (i + 1) % cfg["grad_accum"] == 0 or (i+1)==len(loader):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, cfg["max_grad_norm"])
                scaler.step(opt); scaler.update(); sched.step()
                step += 1
                if on_step is not None:
                    on_step(model, step)
                opt.zero_grad()
        rec = {"epoch": epoch + 1, "train_loss": run / max(1, nb), "sec": round(time.time() - t0)}
        if val_fn is not None:
            rec["val"] = val_fn(model)
            if rec["val"] > best:
                best = rec["val"]
                if save_path:
                    torch.save(trainable_state(model), save_path)
        history.append(rec)
        print(rec)
        if last_path:
            tmp = last_path + ".tmp"
            torch.save({"epoch": epoch + 1, "model": trainable_state(model), "opt": opt.state_dict(), "sched": sched.state_dict(),
                        "scaler": scaler.state_dict(), "best": best, "history": history, "step": step,
                        "extra": extra_state[0]() if extra_state is not None else None}, tmp)
            os.replace(tmp, last_path)
    if val_fn is not None and save_path:
        model.load_state_dict(torch.load(save_path, map_location=device), strict=False)
        print(f"restored best epoch (validation score {best:.4f})")
    return history


GEN_GREEDY = dict(max_new_tokens=256,min_new_tokens=8,repetition_penalty=1.05,no_repeat_ngram_size=4,do_sample=False,temperature=1.0,top_k=50,top_p=0.95)

class NgramBlocker:
    def __init__(self, n):
        self.n, self.hist, self.table = n, [], {}

    def add(self, tok):
        self.hist.append(tok)
        if self.n >= 2 and len(self.hist) >= self.n:
            self.table.setdefault(tuple(self.hist[-self.n:-1]), set()).add(tok)

    def banned(self):
        if self.n < 2 or len(self.hist) < self.n - 1:
            return []
        return list(self.table.get(tuple(self.hist[-(self.n - 1):]), ()))


def supports_cache(model):
    p = inspect.signature(model.forward).parameters
    return all(k in p for k in ("attention_mask", "past_key_values", "use_cache", "last_only"))


def _sample_next(logits, temperature, top_k, top_p):
    logits = logits / max(temperature, 1e-5)
    if top_k:
        v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        logits = logits.masked_fill(logits < v[:, [-1]], -float("inf"))
    probs = torch.softmax(logits, dim=-1)
    if top_p:
        sp, si = torch.sort(probs, descending=True, dim=-1)
        sp = sp.masked_fill(torch.cumsum(sp, dim=-1) - sp > top_p, 0.0)
        sp = sp / sp.sum(dim=-1, keepdim=True)
        return torch.gather(si, -1, torch.multinomial(sp, 1))
    return torch.multinomial(probs, 1)


def _next_tokens(logits, gens, blockers, finished, step, eos_id, gen):
    logits = logits.float()
    rp, mn = gen.get("repetition_penalty", 1.0), gen.get("min_new_tokens", 0)
    for b in range(logits.size(0)):
        if finished[b]:
            continue
        if rp != 1.0 and gens[b]:
            idx = torch.tensor(sorted(set(gens[b])), device=logits.device)
            v = logits[b, idx]
            logits[b, idx] = torch.where(v > 0, v / rp, v * rp)
        ban = blockers[b].banned()
        if ban:
            logits[b, torch.tensor(ban, device=logits.device)] = -float("inf")
        if step < mn:
            logits[b, eos_id] = -float("inf")
    if gen.get("do_sample", False):
        return _sample_next(logits, gen.get("temperature", 1.0), gen.get("top_k", 0), gen.get("top_p", 0.0)).squeeze(-1)
    return logits.argmax(dim=-1)


@torch.no_grad()
def generate_ids(model, prompts, gen, pad_id, eos_id, device, use_amp=True):
    model.eval()
    amp = use_amp and device.type == "cuda"
    ctx = getattr(model, "context_length", None)
    max_new = gen["max_new_tokens"]
    assert ctx is None or max(len(p) for p in prompts) + max_new <= ctx, "prompt + max_new_tokens exceed the context length"
    B, T0 = len(prompts), max(len(p) for p in prompts)
    ids = torch.full((B, T0), pad_id, dtype=torch.long)
    mask = torch.zeros((B, T0), dtype=torch.long)
    for i, p in enumerate(prompts):
        ids[i, T0 - len(p):] = torch.tensor(p)
        mask[i, T0 - len(p):] = 1
    ids, mask = ids.to(device), mask.to(device)
    gens, blockers, finished = [[] for _ in range(B)], [NgramBlocker(gen.get("no_repeat_ngram_size", 0)) for _ in range(B)], [False] * B
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
        out = model(ids, attention_mask=mask, use_cache=True, last_only=True)
    past, logits = out["past_key_values"], out["logits"][:, -1, :]
    for step in range(max_new):
        nxt = _next_tokens(logits, gens, blockers, finished, step, eos_id, gen)
        for b in range(B):
            if finished[b]:
                continue
            t = int(nxt[b])
            if t == eos_id:
                finished[b] = True
            else:
                gens[b].append(t); blockers[b].add(t)
        if all(finished) or step == max_new - 1:
            break
        feed = torch.where(torch.tensor(finished, device=device), torch.full_like(nxt, eos_id), nxt).unsqueeze(1)
        mask = torch.cat([mask, torch.ones((B, 1), dtype=mask.dtype, device=device)], dim=1)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
            out = model(feed, attention_mask=mask, past_key_values=past, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]
    return gens


@torch.no_grad()
def generate_ids_nocache(model, prompts, gen, pad_id, eos_id, device, use_amp=True):
    model.eval()
    amp = use_amp and device.type == "cuda"
    ctx = getattr(model, "context_length", None)
    results = []
    for p in prompts:
        seq = torch.tensor([p], device=device)
        gens, blocker, finished = [[]], [NgramBlocker(gen.get("no_repeat_ngram_size", 0))], [False]
        for step in range(gen["max_new_tokens"]):
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                logits = get_logits(model(seq))[:, -1, :]
            t = int(_next_tokens(logits, gens, blocker, finished, step, eos_id, gen)[0])
            if t == eos_id:
                break
            gens[0].append(t); blocker[0].add(t)
            seq = torch.cat([seq, torch.tensor([[t]], device=device)], dim=1)
        results.append(gens[0])
    return results


def generate_summaries(model, examples, gen, D, device, batch_size=4, use_amp=True, desc="generating"):
    model.eval()
    cache = supports_cache(model)
    bs = batch_size if cache else 1
    order = sorted(range(len(examples)), key=lambda i: len(examples[i]["prompt"]))    
    outs = [None] * len(examples)
    fn = generate_ids if cache else generate_ids_nocache
    for s in tqdm(range(0, len(order), bs), desc=desc, leave=False):
        idxs = order[s:s + bs]
        gens = fn(model, [examples[i]["prompt"].tolist() for i in idxs], gen, D["pad_id"], D["eos_id"], device, use_amp)
        for i, g in zip(idxs, gens):
            outs[i] = D["decode"](g).strip()
    return outs


@torch.no_grad()
def check_generation(model, D, device, n_new=12):
    if not supports_cache(model):
        print("this model has no KV-cache interface: the recompute path is used, nothing to compare")
        return
    model.to(device).eval()
    gen = dict(max_new_tokens=n_new, min_new_tokens=n_new, no_repeat_ngram_size=0, do_sample=False)
    rng = np.random.default_rng(0)
    prompts = [rng.integers(4, D["vocab_size"], size=n).tolist() for n in (40, 25)]      
    a = generate_ids(model, prompts, gen, D["pad_id"], D["eos_id"], device, use_amp=False)
    b = generate_ids_nocache(model, prompts, gen, D["pad_id"], D["eos_id"], device, use_amp=False)
    print("cache path == recompute path:", a == b)
    if a != b:
        print("cache  :", a); print("  nocache:", b)

def _words(s):
    return re.findall(r"[a-z0-9]+", s.lower())


def _f1(p, r):
    return 2 * p * r / (p + r) if p + r else 0.0


def rouge_n_f1(ref, hyp, n):
    rc, hc = Counter(zip(*[ref[i:] for i in range(n)])), Counter(zip(*[hyp[i:] for i in range(n)]))
    ov = sum((rc & hc).values())
    return _f1(ov / max(1, sum(hc.values())), ov / max(1, sum(rc.values())))


def lcs_len(a, b):
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return prev[-1]


def rouge_l_f1(ref, hyp):
    if not ref or not hyp:
        return 0.0
    l = lcs_len(ref, hyp)
    return _f1(l / len(hyp), l / len(ref))


def rouge_scores(refs, hyps):
    r1, r2, rl = [], [], []
    for ref, hyp in zip(refs, hyps):
        a, b = _words(ref), _words(hyp)
        r1.append(rouge_n_f1(a, b, 1)); r2.append(rouge_n_f1(a, b, 2)); rl.append(rouge_l_f1(a, b))
    return {"rouge1": float(np.mean(r1)), "rouge2": float(np.mean(r2)), "rougeL": float(np.mean(rl))}


def summary_stats(hyps, refs=None):
    lens, rep = [], []
    for h in hyps:
        w = _words(h)
        lens.append(len(w))
        tri = list(zip(w, w[1:], w[2:]))
        rep.append(1 - len(set(tri)) / len(tri) if tri else 0.0)
    out = {"avg_words": float(np.mean(lens)), "repeated_trigram_rate": float(np.mean(rep)), "empty": int(sum(l == 0 for l in lens))}
    if refs is not None:
        out["ref_avg_words"] = float(np.mean([len(_words(r)) for r in refs]))
    return out


def evaluate_generation(model, val_examples, gen, D, device, n=None, batch_size=4):
    ex = val_examples[:n] if n else val_examples
    hyps = generate_summaries(model, ex, gen, D, device, batch_size)
    refs = [e["ref"] for e in ex]
    sc, st = rouge_scores(refs, hyps), summary_stats(hyps, refs)
    print(f"ROUGE-1 {sc['rouge1']:.4f} | ROUGE-2 {sc['rouge2']:.4f} | ROUGE-L {sc['rougeL']:.4f} "f"words {st['avg_words']:.0f} (reference {st['ref_avg_words']:.0f}) | repeated trigrams {st['repeated_trigram_rate']:.3f} | empty {st['empty']}")
    return sc, st, hyps

def generate_for_cases(model, raw_docs, D, DOC, gen, device, batch_size=4, fallback_words=60):
    pairs = normalize_pairs(raw_docs)
    docs = tokenize_pairs(pairs, D["encode"], D["clean"], D["max_doc_tokens"], desc="tokenizing documents")
    sums = generate_summaries(model, build_examples(docs, DOC, D, with_target=False), gen, D, device, batch_size)
    clean = D["clean"] or (lambda x: x)
    out = []
    for p, s in zip(pairs, sums):
        if not s.strip():
            s = " ".join(clean(p["document"]).split()[:fallback_words]) or "."
        out.append({"id": p["id"], "summary": s})
    return out


def write_summ_submission(preds, out_dir="/kaggle/working", zip_name="submission_summ.zip"):
    ids = [p["id"] for p in preds]
    path = os.path.join(out_dir, "prediction.json")
    json.dump(preds, open(path, "w", encoding="utf-8"))
    zp = os.path.join(out_dir, zip_name)
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    print(f"wrote {zp}: {len(preds)} summaries, {np.mean([len(p['summary'].split()) for p in preds]):.0f} words on average")

In [ ]:
P6 = "Phase_6_Summ"
hf_dataset_repo = "Exploration-Lab/CS-781-Capstone"
p6 = download_summ_files(hf_dataset_repo,P6,HF_TOKEN)     
D = setup_data(
    p6,
    "/kaggle/working/legal_bpe_tokenizer.pkl",
    sep_token="<s>", eos_token="</s>",       
    val_size=100, seed=0,                    
    cache="/kaggle/working/summ_tok.pkl",
)
PAD_ID, VOCAB_SIZE = D["pad_id"], D["vocab_size"]

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.0):
        super().__init__()
        assert isinstance(base, nn.Linear)
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))           
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)

    @torch.no_grad()
    def to_linear(self):
        W = self.base.weight + self.scale * (self.B @ self.A)
        lin = nn.Linear(self.base.in_features, self.base.out_features, bias=self.base.bias is not None).to(W.device, W.dtype)
        lin.weight.copy_(W)
        if self.base.bias is not None:
            lin.bias.copy_(self.base.bias)
        return lin


def list_linears(root):
    for n, m in root.named_modules():
        if isinstance(m, nn.Linear):
            print(f"{n:45s} {m.in_features:5d} -> {m.out_features:5d}  {'bias' if m.bias is not None else 'no-bias'}")


def inject_lora(root, targets=("q_proj", "k_proj", "v_proj", "c_proj", "c_fc"), r=16, alpha=32, dropout=0.1):
    names = [n for n, m in root.named_modules()if isinstance(m, nn.Linear) and any(n == t or n.endswith("." + t) for t in targets)]
    for n in names:
        parent_path, _, child = n.rpartition(".")
        parent = root.get_submodule(parent_path) if parent_path else root
        setattr(parent, child, LoRALinear(getattr(parent, child), r, alpha, dropout))
    print(f"LoRA r={r}, alpha={alpha}: wrapped {len(names)} layers, e.g. {names[:3]}")
    return names


def freeze_for_lora(model):
    for p in model.parameters():
        p.requires_grad = False
    for m in model.modules():
        if isinstance(m, LoRALinear):
            m.A.requires_grad = True
            m.B.requires_grad = True


@torch.no_grad()
def merge_lora(root):
    for _, m in list(root.named_modules()):
        for cn, c in list(m.named_children()):
            if isinstance(c, LoRALinear):
                setattr(m, cn, c.to_linear())


def run_lora_summ(build_fn, ckpt, D, DOC, device, tag="summ_lora", r=16, alpha=32, targets=("q_proj", "k_proj", "v_proj", "c_proj", "c_fc"),dropout=0.1, cfg=None, resume=False):

    cfg = cfg or CFG_SUMM
    model = load_phase1(build_fn, ckpt)
    inject_lora(model, targets, r, alpha, dropout)
    freeze_for_lora(model)
    count_params(model)
    train_items = to_train_items(build_examples(D["train"], DOC, D))
    val_items = to_train_items(build_examples(D["val"], DOC, D))
    val_fn = lambda m: -evaluate_loss(m, val_items, D["pad_id"], device)
    hist = train_summ(model, train_items, D["pad_id"], cfg, device, val_fn=val_fn,save_path=f"/kaggle/working/{tag}_best.pt", resume=resume)
    return model, hist

In [ ]:
class AdaLoRALinear(nn.Module):
    def __init__(self, base, r_init=24, alpha=32, dropout=0.1):
        super().__init__()
        assert isinstance(base, nn.Linear)
        self.base, self.r, self.scale = base, r_init, alpha / r_init
        self.P = nn.Parameter(torch.randn(base.out_features, r_init) * 0.02)
        self.lam = nn.Parameter(torch.zeros(r_init))
        self.Q = nn.Parameter(torch.randn(r_init, base.in_features) * 0.02)
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        h = F.linear(self.drop(x), self.Q) * self.lam
        return self.base(x) + self.scale * F.linear(h, self.P)

    @torch.no_grad()
    def to_linear(self):
        W = self.base.weight + self.scale * (self.P * self.lam) @ self.Q
        lin = nn.Linear(self.base.in_features, self.base.out_features, bias=self.base.bias is not None).to(W.device, W.dtype)
        lin.weight.copy_(W)
        if self.base.bias is not None:
            lin.bias.copy_(self.base.bias)
        return lin


def inject_adalora(root, targets=("q_proj", "k_proj", "v_proj", "c_proj", "c_fc"), r_init=24, alpha=32, dropout=0.1):
    names = [n for n, m in root.named_modules()
             if isinstance(m, nn.Linear) and any(n == t or n.endswith("." + t) for t in targets)]
    assert names, f"no nn.Linear matched {targets}; run list_linears(model)"
    for n in names:
        parent_path, _, child = n.rpartition(".")
        parent = root.get_submodule(parent_path) if parent_path else root
        setattr(parent, child, AdaLoRALinear(getattr(parent, child), r_init, alpha, dropout))
    print(f"AdaLoRA r_init={r_init}, alpha={alpha}: wrapped {len(names)} layers, e.g. {names[:3]}")
    return names


def freeze_for_adalora(model):
    for p in model.parameters():
        p.requires_grad = False
    for m in model.modules():
        if isinstance(m, AdaLoRALinear):
            for p in (m.P, m.lam, m.Q):
                p.requires_grad = True


@torch.no_grad()
def merge_adalora(root):
    for _, m in list(root.named_modules()):
        for cn, c in list(m.named_children()):
            if isinstance(c, AdaLoRALinear):
                setattr(m, cn, c.to_linear())


class AdaLoRAController:
    def __init__(self, model, target_rank, total_steps, tinit_frac=0.1, tfinal_frac=0.3, delta_t=10,
                 beta1=0.85, beta2=0.85, reg=0.1):
        self.mods = [(n, m) for n, m in model.named_modules() if isinstance(m, AdaLoRALinear)]
        self.b0 = sum(m.r for _, m in self.mods)
        self.bT = int(target_rank * len(self.mods))
        self.total = total_steps
        self.ti, self.tf = int(tinit_frac * total_steps), int(tfinal_frac * total_steps)
        self.dt, self.b1, self.b2, self.reg = delta_t, beta1, beta2, reg
        self.avg, self.unc = {}, {}
        print(f"AdaLoRA budget: {self.b0} -> {self.bT} triplets over {total_steps} steps"f"(init phase {self.ti}, final phase {self.tf}, prune every {delta_t})")

    def reg_loss(self, model):
        tot = 0.0
        for _, m in self.mods:
            I = torch.eye(m.r, device=m.P.device, dtype=m.P.dtype)
            tot = tot + (m.P.T @ m.P - I).norm() + (m.Q @ m.Q.T - I).norm()
        return self.reg * tot / len(self.mods)

    @torch.no_grad()
    def _update_ipt(self):
        for n, m in self.mods:
            for k, p in (("P", m.P), ("lam", m.lam), ("Q", m.Q)):
                if p.grad is None:
                    continue
                key = f"{n}.{k}"
                I = (p * p.grad).abs()
                if key not in self.avg:
                    self.avg[key], self.unc[key] = torch.zeros_like(I), torch.zeros_like(I)
                self.avg[key].mul_(self.b1).add_(I, alpha=1 - self.b1)
                self.unc[key].mul_(self.b2).add_((I - self.avg[key]).abs(), alpha=1 - self.b2)

    def _budget(self, step):
        if step <= self.ti:
            return self.b0
        if step > self.total - self.tf:
            return self.bT
        frac = (step - self.ti) / max(1, self.total - self.tf - self.ti)
        return int(self.bT + (self.b0 - self.bT) * (1 - frac) ** 3)

    @torch.no_grad()
    def _prune(self, budget):
        scores = []
        for n, m in self.mods:
            s = lambda k: self.avg[f"{n}.{k}"] * self.unc[f"{n}.{k}"]
            scores.append(s("lam") + s("P").mean(0) + s("Q").mean(1))
        allsc = torch.cat(scores)
        keep = torch.zeros_like(allsc, dtype=torch.bool)
        keep[torch.topk(allsc, min(budget, allsc.numel())).indices] = True
        o = 0
        for _, m in self.mods:
            m.lam.mul_(keep[o:o + m.r].to(m.lam.dtype))
            o += m.r

    def on_step(self, model, step):
        self._update_ipt()
        if self.ti < step <= self.total - self.tf and step % self.dt == 0:
            self._prune(self._budget(step))

    def ranks(self):
        return {n: int((m.lam.detach().abs() > 0).sum()) for n, m in self.mods}

    def get_state(self):
        return {"avg": {k: v.cpu() for k, v in self.avg.items()}, "unc": {k: v.cpu() for k, v in self.unc.items()}}

    def set_state(self, st):
        dev = self.mods[0][1].P.device
        self.avg = {k: v.to(dev) for k, v in st["avg"].items()}
        self.unc = {k: v.to(dev) for k, v in st["unc"].items()}

CFG_ADALORA = dict(CFG_SUMM)      

def run_adalora_summ(build_fn, ckpt, D, DOC, device, tag="summ_adalora", r_init=24, target_rank=16, alpha=32,targets=("q_proj", "k_proj", "v_proj", "c_proj", "c_fc"), dropout=0.1, reg=0.1, delta_t=10,cfg=None, resume=False):
    cfg = cfg or CFG_ADALORA
    model = load_phase1(build_fn, ckpt)
    inject_adalora(model, targets, r_init, alpha, dropout)
    freeze_for_adalora(model)
    count_params(model)
    model.to(device)
    train_items = to_train_items(build_examples(D["train"], DOC, D))
    val_items = to_train_items(build_examples(D["val"], DOC, D))
    n_batches = math.ceil(len(train_items) / cfg["batch_size"])
    total_steps = max(1, (n_batches // cfg["grad_accum"]) * cfg["epochs"])         
    ctrl = AdaLoRAController(model, target_rank, total_steps, delta_t=delta_t, reg=reg)
    val_fn = lambda m: -evaluate_loss(m, val_items, D["pad_id"], device)
    hist = train_summ(model, train_items, D["pad_id"], cfg, device, val_fn=val_fn,save_path=f"/kaggle/working/{tag}_best.pt", extra_loss=ctrl.reg_loss, on_step=ctrl.on_step,resume=resume, extra_state=(ctrl.get_state, ctrl.set_state))
    r = ctrl.ranks()
    print(f"final kept triplets {sum(r.values())} (budget {ctrl.bT}); per-module rank, first 8:", list(r.items())[:8])
    return model, hist, ctrl

In [ ]:
class LoRMALinear(nn.Module):
    def __init__(self, base, r=16, alpha=16):
        super().__init__()
        assert isinstance(base, nn.Linear)
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        z = F.linear(x, self.base.weight.to(x.dtype))                                  
        y = z + self.scale * F.linear(F.linear(z, self.A), self.B)
        return y + self.base.bias.to(y.dtype) if self.base.bias is not None else y

    @torch.no_grad()
    def to_linear(self):
        W0 = self.base.weight
        W = W0 + self.scale * (self.B @ (self.A @ W0))                    
        lin = nn.Linear(self.base.in_features, self.base.out_features, bias=self.base.bias is not None).to(W.device, W.dtype)
        lin.weight.copy_(W)
        if self.base.bias is not None:
            lin.bias.copy_(self.base.bias)
        return lin


def inject_lorma(root, targets=("q_proj", "k_proj", "v_proj", "c_proj", "c_fc"), r=16, alpha=16):
    names = [n for n, m in root.named_modules()
             if isinstance(m, nn.Linear) and any(n == t or n.endswith("." + t) for t in targets)]
    assert names, f"no nn.Linear matched {targets}; run list_linears(model)"
    for n in names:
        parent_path, _, child = n.rpartition(".")
        parent = root.get_submodule(parent_path) if parent_path else root
        setattr(parent, child, LoRMALinear(getattr(parent, child), r, alpha))
    print(f"LoRMA r={r}, alpha={alpha}: wrapped {len(names)} layers, e.g. {names[:3]}")
    return names


def freeze_for_lorma(model):
    for p in model.parameters():
        p.requires_grad = False
    for m in model.modules():
        if isinstance(m, LoRMALinear):
            m.A.requires_grad = True
            m.B.requires_grad = True


@torch.no_grad()
def merge_lorma(root):
    for _, m in list(root.named_modules()):
        for cn, c in list(m.named_children()):
            if isinstance(c, LoRMALinear):
                setattr(m, cn, c.to_linear())


def run_lorma_summ(build_fn, ckpt, D, DOC, device, tag="summ_lorma", r=16, alpha=16,targets=("q_proj", "k_proj", "v_proj", "c_proj", "c_fc"), cfg=None, resume=False):
    cfg = cfg or CFG_SUMM
    model = load_phase1(build_fn, ckpt)
    inject_lorma(model, targets, r, alpha)
    freeze_for_lorma(model)
    count_params(model)
    train_items = to_train_items(build_examples(D["train"], DOC, D))
    val_items = to_train_items(build_examples(D["val"], DOC, D))
    val_fn = lambda m: -evaluate_loss(m, val_items, D["pad_id"], device)
    hist = train_summ(model, train_items, D["pad_id"], cfg, device, val_fn=val_fn,save_path=f"/kaggle/working/{tag}_best.pt", resume=resume)
    return model, hist

In [ ]:
class KDWrapper(nn.Module):
    def __init__(self, student, teacher, sep_id, pad_id, T=2.0):
        super().__init__()
        self.student, self.teacher = student, teacher
        for p in self.teacher.parameters():
            p.requires_grad = False
        self.sep_id, self.pad_id, self.T = sep_id, pad_id, T
        self.kd = None

    def train(self, mode=True):
        super().train(mode)
        self.teacher.eval()                                  
        return self

    def forward(self, ids):
        ls = get_logits(self.student(ids))
        if self.training:
            with torch.no_grad():
                lt = get_logits(self.teacher(ids))
            L = ids.size(1)
            last_sep = L - 1 - (ids == self.sep_id).flip(1).float().argmax(1)          
            pos = torch.arange(L - 1, device=ids.device)[None]
            m = (pos >= last_sep[:, None]) & (ids[:, 1:] != self.pad_id)
            s = ls[:, :-1][m].float() / self.T
            t = lt[:, :-1][m].float() / self.T
            self.kd = F.kl_div(F.log_softmax(s, -1), F.softmax(t, -1), reduction="batchmean") * self.T ** 2
        return ls


def load_teacher_from_lora(build_fn, ckpt, lora_best_path, device, targets=("q_proj", "k_proj", "v_proj", "c_proj", "c_fc"),r=16, alpha=32):
    t = load_phase1(build_fn, ckpt)
    inject_lora(t, targets, r, alpha)
    t.load_state_dict(torch.load(lora_best_path, map_location="cpu"), strict=False)
    merge_lora(t)
    t.to(device).eval()
    for p in t.parameters():
        p.requires_grad = False
    return t


CFG_KD = dict(CFG_SUMM)              

def run_kd_summ(build_student, student_ckpt, teacher, D, DOC, device, tag="summ_kd", T=2.0, kd_weight=1.0,cfg=None, resume=False, train_items=None):
    cfg = cfg or CFG_KD
    student = build_student() if student_ckpt is None else load_phase1(build_student, student_ckpt)
    for p in student.parameters():
        p.requires_grad = True
    count_params(student)
    teacher.to(device).eval()
    wrap = KDWrapper(student, teacher, D["sep_id"], D["pad_id"], T).to(device)
    if train_items is None:
        train_items = to_train_items(build_examples(D["train"], DOC, D))
    val_items = to_train_items(build_examples(D["val"], DOC, D))
    val_fn = lambda m: -evaluate_loss(m.student, val_items, D["pad_id"], device)       
    hist = train_summ(wrap, train_items, D["pad_id"], cfg, device, val_fn=val_fn,save_path=f"/kaggle/working/{tag}_best.pt", extra_loss=lambda m: kd_weight * m.kd, resume=resume)
    return wrap.student, hist


In [ ]:
CFG_SUMM = dict(epochs=8, batch_size=2, grad_accum=8, lr=1e-4, weight_decay=0.01, warmup_ratio=0.05,max_grad_norm=1.0, fp16=True, seed=0)    

In [ ]:
DOC = DOC_SPREAD                            
model_lora, hist_lora = run_lora_summ(build_model, MODEL_PT, D, DOC, device, tag="summ_lora_v2.2")

paths = download_summ_files(include_test=True)          
test_raw = json.load(open(paths["test"], encoding="utf-8"))
preds = generate_for_cases(model_lora, test_raw, D, DOC, GEN_GREEDY, device, batch_size=4)
write_summ_submission(preds, out_dir="/kaggle/working", zip_name="submission_summ_lora_v2.2.zip")

In [ ]:
DOC = DOC_SPREAD
model_adalora, hist_adalora, ctrl_adalora = run_adalora_summ(build_model, MODEL_PT, D, DOC, device, tag="summ_adalora_v1",r_init=24, target_rank=16)

paths = download_summ_files(hf_dataset_repo, P6, HF_TOKEN, include_test=True)
test_raw = json.load(open(paths["test"], encoding="utf-8"))
preds = generate_for_cases(model_adalora, test_raw, D, DOC, GEN_GREEDY, device, batch_size=4)
write_summ_submission(preds, out_dir="/kaggle/working", zip_name="submission_summ_adalora.zip")

In [ ]:
DOC = DOC_SPREAD
model_lorma, hist_lorma = run_lorma_summ(build_model, MODEL_PT, D, DOC, device, tag="summ_lorma_v1", r=16, alpha=16)

sc, st, hyps = evaluate_generation(model_lorma, build_examples(D["val"], DOC, D), GEN_GREEDY, D, device, n=50)
paths = download_summ_files(hf_dataset_repo, P6, HF_TOKEN, include_test=True)
test_raw = json.load(open(paths["test"], encoding="utf-8"))
preds = generate_for_cases(model_lorma, test_raw, D, DOC, GEN_GREEDY, device, batch_size=4)
write_summ_submission(preds, out_dir="/kaggle/working", zip_name="submission_summ_lorma.zip")

In [ ]:
build_student = build_model
STUDENT_PT = MODEL_PT

teacher = load_teacher_from_lora(build_model, MODEL_PT, "/kaggle/working/summ_lora_v1_best.pt", device,r=16, alpha=32)               
DOC = DOC_SPREAD
student, hist = run_kd_summ(build_student, STUDENT_PT, teacher, D, DOC, device, tag="summ_kd_v1", T=2.0, kd_weight=1.0)

sc, st, hyps = evaluate_generation(student, build_examples(D["val"], DOC, D), GEN_GREEDY, D, device, n=50)
paths = download_summ_files(hf_dataset_repo, P6, HF_TOKEN, include_test=True)
test_raw = json.load(open(paths["test"], encoding="utf-8"))
preds = generate_for_cases(student, test_raw, D, DOC, GEN_GREEDY, device, batch_size=4)
write_summ_submission(preds, out_dir="/kaggle/working", zip_name="submission_summ_kd.zip")